# 01 · From Python to image pixels

**Level:** first steps · **Time:** 35–50 minutes · **Data:** bundled handwritten digits, no download.

By the end you can use variables, loops and functions; inspect a NumPy image; plot it;
and explain the difference between a pixel, an image and a class label.
**Prerequisite:** open this notebook using the course kernel. No machine-learning experience needed.

Run each cell with **Shift+Enter**, then change one value and rerun it. A notebook remembers
variables in its kernel: after large edits, use **Restart Kernel and Run All** to check the full sequence.
Blue/black text cells explain ideas; code cells execute Python. Errors are clues: read the last line first.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the ObjectDetectionPractical folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, get_device, SMOKE_TEST, ARTIFACTS
seed_everything(42)
device = get_device()
print("Device:", device, "| smoke test:", SMOKE_TEST)


## Step 1 — Give values names
A variable keeps a value for later. Python infers its type: integers count things,
floats hold decimal values, strings hold text, and booleans are `True` or `False`.
A list stores an ordered sequence. Indexing starts at **zero**.


In [ ]:
image_height = 8                 # An integer
pixel_fraction = 0.5             # A float
dataset_name = "handwritten digits"
is_grayscale = True              # One intensity channel
class_names = [str(number) for number in range(10)]
print(dataset_name, "|", image_height, "pixels high")
print("First class:", class_names[0], "| last class:", class_names[-1])
print(type(image_height).__name__, type(pixel_fraction).__name__)


## Step 2 — Repeat with a loop, reuse with a function
Indentation marks which statements belong together. A `for` loop visits each value.
A function accepts inputs and returns a result. Comments begin with `#` and do not run.
Here, a conditional (`if`) makes a decision about each intensity.


In [ ]:
def describe_intensity(value):
    if value < 0.5:
        return "dark"
    return "bright"

for value in [0.0, 0.25, 0.75, 1.0]:
    print(value, "is", describe_intensity(value))


## Step 3 — Load images and labels
`load_digits` contains 1,797 tiny grayscale images of digits 0–9. Each image has
8 rows × 8 columns. Its original intensities run from 0 to 16; its **label** tells us the digit.
This classroom dataset is intentionally small. High scores here do not establish reliability
on phone photographs, new handwriting styles or real applications.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.datasets import load_digits

digits = load_digits()
raw_images = digits.images
labels = digits.target
print("Images:", raw_images.shape, "| labels:", labels.shape)
print("Pixel range:", raw_images.min(), "to", raw_images.max())
print("Classes:", np.unique(labels))


## Step 4 — Understand shape, dtype and indexing
A NumPy array is a grid of values with one shared numeric type (`dtype`).
Our shape is `(number_of_images, height, width)`. `images[0]` selects an image;
`images[0, 2, 3]` selects its third row and fourth column.
`:4` selects the first four entries. Make a `.copy()` before changing a slice
if you want to preserve the original array.


In [ ]:
image_index = 0
first_image = raw_images[image_index].copy()
print("True digit:", labels[image_index])
print("Image shape:", first_image.shape, "| dtype:", first_image.dtype)
print("Top-left 4 by 4 pixels:\n", first_image[:4, :4])
print("Pixel at row 2, column 3:", first_image[2, 3])


## Step 5 — Normalize a known intensity range
Divide every pixel by 16 to map this dataset's documented range to `[0, 1]`.
`float32` uses 32 bits per number and is common for neural-network inputs.
Array arithmetic applies to every element: no Python loop is needed.
This fixed rescaling is different from **learning** a mean or variance from data;
learned preprocessing must later be fitted on training examples only.


In [ ]:
images = raw_images.astype(np.float32) / 16.0
print("Normalized:", images.dtype, images.min(), images.max())
print("Mean intensity of first image:", round(float(images[0].mean()), 3))
assert images.shape == (len(labels), 8, 8)
assert 0 <= images.min() and images.max() <= 1


## Step 6 — See the numbers as pictures
`imshow` assigns a color to each number; `cmap="gray"` uses black for low intensity
and white for high intensity. Fixing `vmin` and `vmax` lets panels use the same brightness scale.
Expect blocky but mostly recognizable handwritten digits.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for index, ax in enumerate(axes.flat):
    ax.imshow(images[index], cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(f"Label: {labels[index]}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## Step 7 — Convert between an array and a Pillow image
Pillow reads and writes common image formats. An 8-bit grayscale PNG stores integer
intensities 0–255. Resize with nearest-neighbor sampling to make the existing pixels
easy to inspect; resizing does **not** recover new image detail.
RGB photographs usually have shape `(height, width, 3)` in NumPy.


In [ ]:
pil_digit = Image.fromarray((images[0] * 255).round().astype(np.uint8))
enlarged = pil_digit.resize((160, 160), resample=Image.Resampling.NEAREST)
rgb_digit = pil_digit.convert("RGB")
output_dir = ARTIFACTS / "01_pixels"
output_dir.mkdir(parents=True, exist_ok=True)
image_path = output_dir / "first_digit.png"
enlarged.save(image_path)
with Image.open(image_path) as reopened:
    reopened_array = np.asarray(reopened).copy()
print("Pillow mode:", pil_digit.mode, "| RGB array shape:", np.asarray(rgb_digit).shape)
print("Saved and reopened:", reopened_array.shape, image_path)
plt.imshow(reopened_array, cmap="gray", vmin=0, vmax=255)
plt.axis("off")
plt.show()


## Step 8 — An image can become a feature vector
Flattening an 8 × 8 grid creates 64 features in a fixed order. A traditional classifier
can accept a table shaped `(examples, features)`. The values survive the reshape,
but the model must learn useful relationships between positions.
Convolutional networks later exploit the spatial grid directly.


In [ ]:
feature_table = images.reshape(len(images), -1)
restored = feature_table[0].reshape(8, 8)
print("Feature table:", feature_table.shape)
print("Reshape preserved the pixels:", np.allclose(restored, images[0]))
fig, ax = plt.subplots(figsize=(7, 2))
ax.plot(feature_table[0], marker=".")
ax.set(xlabel="Flattened pixel position", ylabel="Intensity", ylim=(-0.05, 1.05))
plt.show()


## Practice 1 — Inspect a different digit
Change `chosen_index` to 27. Print its label, mean and brightest pixel; show the image.
Predict whether its average brightness will be closer to 0 or 1 before running.

<details><summary>Sample answer — open after trying</summary>

```python
chosen_index = 27
chosen = images[chosen_index]
print(labels[chosen_index], chosen.mean(), chosen.max())
plt.imshow(chosen, cmap="gray", vmin=0, vmax=1)
plt.show()
```

Most pixels are dark background, so the average is usually closer to zero.
</details>


In [ ]:
chosen_index = 12  # Change this; keep it between 0 and len(images) - 1.
chosen = images[chosen_index]
print("Your selected label:", labels[chosen_index])


## Practice 2 — Write a brightness function
Write a function that returns the fraction of pixels brighter than a threshold.
Apply it to three images using a loop. Hint: the mean of a boolean array is its fraction of `True` values.

<details><summary>Sample answer</summary>

```python
def bright_fraction(image, threshold=0.5):
    return float((image > threshold).mean())
for index in [0, 1, 2]:
    print(index, bright_fraction(images[index]))
```

`>` compares each pixel and produces a boolean mask with the same shape.
</details>


In [ ]:
threshold = 0.5
bright_mask = images[0] > threshold
print("Starter mask shape:", bright_mask.shape)
# Add your function and loop below.


## Practice 3 — Change pixels without changing your original
Make an inverted copy, then make a copy with the top two rows black. Plot both.
Explain why a label is separate from its pixel values: an edit can make an image ambiguous
even when you keep its old label.

<details><summary>Sample answer</summary>

```python
inverted = 1.0 - images[0]
cropped_ink = images[0].copy()
cropped_ink[:2, :] = 0
fig, axes = plt.subplots(1, 2)
for ax, image in zip(axes, [inverted, cropped_ink]):
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
plt.show()
```

The label is supplied as supervision; the array itself contains no embedded class name.
</details>


In [ ]:
edited_image = images[0].copy()
print("Original still unchanged:", np.allclose(edited_image, images[0]))
# Make and display your edits below.


## What you can now do
You used Python variables, indexing, conditionals, loops and functions; inspected array shapes
and dtypes; normalized, displayed, reshaped and saved an image. These operations are the
input pipeline for a classifier. No classifier was trained in this lesson.

**Checkpoint:** explain `(1797, 8, 8)` aloud, and explain why the number `7` as a label
differs from a pixel whose intensity is `7`.

**Next:** [02 · Classical machine-learning baselines](02_classical_ml_baselines.ipynb).

Primary references: [Python tutorial](https://docs.python.org/3/tutorial/),
[NumPy beginner guide](https://numpy.org/doc/stable/user/absolute_beginners.html),
[digits dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html),
[Matplotlib imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html),
[Pillow image API](https://pillow.readthedocs.io/en/stable/reference/Image.html).
